Importing dependencies and loading cleaned data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from gensim.models import Word2Vec

import nltk
nltk.download('stopwords', quiet=True)

train_df = pd.read_csv("Dataset/train.csv")
test_df  = pd.read_csv("Dataset/test.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nColumns:", train_df.columns.tolist())
print("\nLabel distribution (train):")
print(train_df["label"].value_counts().rename({0: "Fake", 1: "True"}))

Train shape: (33043, 6)
Test shape : (11015, 6)

Columns: ['title', 'text', 'cleaned_title', 'cleaned_text', 'content', 'label']

Label distribution (train):
label
Fake    17136
True    15907
Name: count, dtype: int64


TF-IDF: comparing the usage of news body only to title + body with L2 norm

In [3]:
from sklearn.preprocessing import normalize

# Fill any NaN just in case
train_df["cleaned_text"] = train_df["cleaned_text"].fillna("")
train_df["content"]      = train_df["content"].fillna("")
test_df["cleaned_text"]  = test_df["cleaned_text"].fillna("")
test_df["content"]       = test_df["content"].fillna("")

# --- Body-only TF-IDF ---
tfidf_body = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), norm="l2")
X_train_body = tfidf_body.fit_transform(train_df["cleaned_text"])
X_test_body  = tfidf_body.transform(test_df["cleaned_text"])

# --- Title+Body TF-IDF ---
tfidf_combined = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), norm="l2")
X_train_combined = tfidf_combined.fit_transform(train_df["content"])
X_test_combined  = tfidf_combined.transform(test_df["content"])

y_train = train_df["label"]
y_test  = test_df["label"]

print("Body-only matrix    :", X_train_body.shape)
print("Title+Body matrix   :", X_train_combined.shape)
print("L2 norm sample (body row 0):", X_train_body[0].toarray().sum()**0.5)


Body-only matrix    : (33043, 50000)
Title+Body matrix   : (33043, 50000)
L2 norm sample (body row 0): 3.6778509253418203


Using cross validation to tune hyperparameter C and ngram_range

In [4]:
from sklearn.model_selection import cross_val_score

configs = [
    {"name": "body  | (1,1) | C=0.1",  "X": train_df["cleaned_text"], "ngram": (1,1), "C": 0.1},
    {"name": "body  | (1,2) | C=0.1",  "X": train_df["cleaned_text"], "ngram": (1,2), "C": 0.1},
    {"name": "body  | (1,2) | C=1.0",  "X": train_df["cleaned_text"], "ngram": (1,2), "C": 1.0},
    {"name": "body  | (1,2) | C=10.0", "X": train_df["cleaned_text"], "ngram": (1,2), "C": 10.0},
    {"name": "comb  | (1,1) | C=0.1",  "X": train_df["content"],      "ngram": (1,1), "C": 0.1},
    {"name": "comb  | (1,2) | C=0.1",  "X": train_df["content"],      "ngram": (1,2), "C": 0.1},
    {"name": "comb  | (1,2) | C=1.0",  "X": train_df["content"],      "ngram": (1,2), "C": 1.0},
    {"name": "comb  | (1,2) | C=10.0", "X": train_df["content"],      "ngram": (1,2), "C": 10.0},
]

results = []
for cfg in configs:
    vec = TfidfVectorizer(max_features=50000, ngram_range=cfg["ngram"], norm="l2")
    X   = vec.fit_transform(cfg["X"])
    clf = LogisticRegression(C=cfg["C"], max_iter=1000, solver="lbfgs", n_jobs=-1)
    scores = cross_val_score(clf, X, y_train, cv=5, scoring="f1", n_jobs=-1)
    results.append({"config": cfg["name"], "mean_f1": scores.mean(), "std_f1": scores.std()})
    print(f"{cfg['name']}  →  F1: {scores.mean():.4f} ± {scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values("mean_f1", ascending=False)
print("\n--- Ranked ---")
print(results_df.to_string(index=False))


body  | (1,1) | C=0.1  →  F1: 0.9671 ± 0.0034
body  | (1,2) | C=0.1  →  F1: 0.9700 ± 0.0033
body  | (1,2) | C=1.0  →  F1: 0.9862 ± 0.0028
body  | (1,2) | C=10.0  →  F1: 0.9923 ± 0.0025
comb  | (1,1) | C=0.1  →  F1: 0.9671 ± 0.0030
comb  | (1,2) | C=0.1  →  F1: 0.9703 ± 0.0028
comb  | (1,2) | C=1.0  →  F1: 0.9870 ± 0.0028
comb  | (1,2) | C=10.0  →  F1: 0.9930 ± 0.0024

--- Ranked ---
                config  mean_f1   std_f1
comb  | (1,2) | C=10.0 0.992964 0.002445
body  | (1,2) | C=10.0 0.992308 0.002472
 comb  | (1,2) | C=1.0 0.986987 0.002799
 body  | (1,2) | C=1.0 0.986171 0.002810
 comb  | (1,2) | C=0.1 0.970323 0.002755
 body  | (1,2) | C=0.1 0.969985 0.003299
 body  | (1,1) | C=0.1 0.967109 0.003370
 comb  | (1,1) | C=0.1 0.967064 0.003019
